## Validate against Sanger sequencing results

In [110]:
import os
import glob
import pandas as pd

## Settings

In [111]:
save_results = True
DIR_OUTPUT = "../../tables/"

## Load data

In [112]:
# Load samples
df_samples = pd.read_csv(
    "../sample_set/table.zambia_sanger.all_info.csv", dtype={"sample_id":str}
)
eastern_samples = df_samples.query("province == 'eastern'").sample_id.to_list()

# For Eastern Province, no 724E calls, so we need to pull coverage from outside the VCF.
df_coverage = (
    pd.read_csv("../../seqdata/all_standard/summaries/HRP23_MIS2024/summary.coverage.csv",
                dtype=dict(sample_id=str))
    .query("sample_id in @eastern_samples")
    .query("sample_type == 'field'")
    .query("name == 'kelch13-p383-727'")
    .query("status == 'pass'")
)
df_coverage.index = df_coverage.sample_id
coverage = pd.Series(df_coverage['mean_cov'])
df_samples['dp_A724E'] = [d if not pd.isna(d) else coverage[s] for s, d in zip(df_samples['sample_id'], df_samples['dp_A724E'])]
df_samples['dp_A724E'] = df_samples['dp_A724E'].round(0).astype(int)
df_samples['wsaf_A724E'] = df_samples['wsaf_A724E'].round(2).astype(float)

In [113]:
df_samples = df_samples[['sample_id', 'province', 'district', 'dp_A724E', 'wsaf_A724E', 'gt_A724E']]

In [117]:
def load_sangerdata_tsv(assay: str) -> pd.DataFrame:
    # Load
    dfs = []
    for csv in glob.glob(f"../results/{assay}/4tsvs/*.tsv"):
        try:
            df = pd.read_csv(csv, dtype={"sample_id": str}, sep="\t")
        except:
            print(f"Failed to load {csv}!")
            continue
        dfs.append(df)

    # Annotate 724E
    df = pd.concat(dfs)
    df["sanger_A724E"] = ["724A>724E" in csq for csq in df["csq"]]
                          
    return df

In [118]:
def load_sangerdata(assay: str) -> pd.DataFrame:
    # Load
    dfs = []
    for csv in glob.glob(f"../results/{assay}/4csvs/*.csv"):
        try:
            df = pd.read_csv(csv, dtype={"sample_id": str})
        except:
            print(f"Failed to load {csv}!")
            continue
        dfs.append(df)

    # Annotate 724E
    df = pd.concat(dfs)
    df["sanger_A724E"] = ["724A>724E" in csq for csq in df["csq"]]
                          
    return df

In [119]:
call_724e = lambda df, name: (df
    .groupby("sample_id")
    .sanger_A724E.sum()
    .reset_index()
    .rename({'sanger_A724E': f'{name}_A724E'}, axis=1)
)

In [141]:
df_assay1 = load_sangerdata_tsv("assay1")
df_assay2 = load_sangerdata_tsv("assay2")

In [155]:
df_sanger = pd.merge(
    left=call_724e(df_assay1, 'assay1'),
    right=call_724e(df_assay2, 'assay2'),
    how='outer',
    on='sample_id',
    validate='1:1'
)

In [156]:
df_sanger.shape[0], df_samples.shape[0]

(38, 44)

In [167]:
df_merged = pd.merge(left=df_samples,
                     right=df_sanger,
                     how='left',
                     on='sample_id',
                     validate='1:1')
df_merged[['assay1_A724E', 'assay2_A724E']] = df_merged[['assay1_A724E', 'assay2_A724E']].fillna("FAIL")
df_merged['combined_A724E'] = [a1 if a1 != 'FAIL' else a2 for a1, a2 in zip(df_merged['assay1_A724E'],
                                                                              df_merged['assay2_A724E'])]

In [168]:
# Clean
#df_merged['gt_A724E'] = df_merged['gt_A724E'].map({2.0: 'MUT', 1.0: 'MIX', 0.0: 'WT'})
for c in ['gt_A724E', 'assay1_A724E', 'assay2_A724E','combined_A724E']:
    df_merged[c] = pd.Categorical(
        df_merged[c].map({2.0: 'MUT', 1.0: 'MUT', 0.0: 'WT', 'FAIL': 'FAIL'}),
        categories=["MUT", "WT", "FAIL"], ordered=True
)

In [169]:
# Some small cleaning
df_merged["province"] = df_merged["province"].str.capitalize()
df_merged["district"] = df_merged["district"].str.capitalize()

In [170]:
df_merged = df_merged.sort_values(["gt_A724E", "combined_A724E", "assay1_A724E", "assay2_A724E"])

In [182]:
df_merged

,sample_id,province,district,dp_A724E,wsaf_A724E,gt_A724E,assay1_A724E,assay2_A724E,combined_A724E
0,1012420,Northwestern,Mufumbwe,2885,1.00,MUT,MUT,MUT,MUT
3,1012542,Northwestern,Zambezi,528,1.00,MUT,MUT,MUT,MUT
7,1046674,Western,Luampa,81,1.00,MUT,MUT,MUT,MUT
10,5053742,Western,Sikongo,415,1.00,MUT,MUT,MUT,MUT
17,8019456,Northwestern,Kasempa,1940,1.00,MUT,MUT,MUT,MUT
18,8021039,Northwestern,Kasempa,4617,1.00,MUT,MUT,MUT,MUT
19,8021930,Western,Kaoma,224,1.00,MUT,MUT,MUT,MUT
25,8023320,Northwestern,Manyinga,987,1.00,MUT,MUT,MUT,MUT
26,8023351,Northwestern,Kabompo,542,1.00,MUT,MUT,MUT,MUT
27,8023430,Northwestern,Kabompo,664,1.00,MUT,MUT,MUT,MUT


In [183]:
if True:
    df_merged.to_csv(f"{DIR_OUTPUT}/stable_sanger-results.csv", index=False)

## Clean and add final summary tables

In [173]:
pd.crosstab(
    df_merged['assay1_A724E'],
    df_merged['gt_A724E'],
    margins='All'
)

gt_A724E,MUT,WT,All
assay1_A724E,,,
MUT,16,0,16
WT,0,12,12
FAIL,6,10,16
All,22,22,44


In [174]:
pd.crosstab(
    df_merged['assay2_A724E'],
    df_merged['gt_A724E'],
    margins='All'
)

gt_A724E,MUT,WT,All
assay2_A724E,,,
MUT,19,0,19
WT,0,19,19
FAIL,3,3,6
All,22,22,44


In [175]:
pd.crosstab(
    df_merged['assay2_A724E'],
    df_merged['gt_A724E'],
    margins='All'
)

gt_A724E,MUT,WT,All
assay2_A724E,,,
MUT,19,0,19
WT,0,19,19
FAIL,3,3,6
All,22,22,44


In [176]:
pd.crosstab(
    df_merged['assay2_A724E'],
    df_merged['gt_A724E'],
    margins='All'
)

gt_A724E,MUT,WT,All
assay2_A724E,,,
MUT,19,0,19
WT,0,19,19
FAIL,3,3,6
All,22,22,44


In [177]:
df_merged.combined_A724E.value_counts()

combined_A724E
MUT     19
WT      19
FAIL     6
Name: count, dtype: int64

In [178]:
(df_merged.combined_A724E != 'FAIL').sum() / df_merged.shape[0]

np.float64(0.8636363636363636)

In [179]:
if save_results:
    ct_sanger.to_csv(f"{DIR_OUTPUT}/stable_sanger-results-crosstable.csv", index=False)